In [ ]:
import os
os.environ["GROK_API_KEY"] = ""

In [45]:
# !pip install -q langchain-openai langchain-core requests

In [52]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [53]:
# Tool Create 

@tool
def multiple(a: int, b: int) -> int:
    """Return the product"""
    return a * b

In [54]:
print(multiple.invoke({"a": 6, "b": 3}))

18


In [55]:
multiple.name

'multiple'

In [56]:
multiple.description

'Return the product'

In [57]:
multiple.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

'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'

# Tool Binding in Langchain

In [62]:
llm = ChatGroq(model="openai/gpt-120b", api_key=os.getenv("GROK_API_KEY"))

In [63]:
# This is only tool binding implementation.

llm_with_tools = llm.bind_tools([multiple])

In [64]:
result = llm_with_tools.invoke("what is 10 multiply by 3")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `openai/gpt-120b` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [65]:
query = HumanMessage("can you multiply 3 with the 10")

In [14]:
message = [query]

In [15]:
message

[HumanMessage(content='can you multiply 3 with the 10', additional_kwargs={}, response_metadata={})]

In [16]:
result = llm_with_tools.invoke(message)

In [17]:
message.append(result)

In [18]:
message[0]

HumanMessage(content='can you multiply 3 with the 10', additional_kwargs={}, response_metadata={})

In [19]:
result.tool_calls[0]

{'name': 'multiple',
 'args': {'a': 3, 'b': 10},
 'id': 'call_19zlosKiUbhVbJ0iyC8X30Of',
 'type': 'tool_call'}

In [20]:
# Tool Execution

tool_result = multiple.invoke(result.tool_calls[0])

In [21]:
message.append(tool_result)

In [22]:
message

[HumanMessage(content='can you multiply 3 with the 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_19zlosKiUbhVbJ0iyC8X30Of', 'function': {'arguments': '{"a": 3, "b": 10}', 'name': 'multiple'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 63, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CCObEUte1cQvTkslY0yvpkwerrfsi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--d9d67e10-9307-4740-956c-4d0bca3b1931-0', tool_calls=[{'name': 'multiple', 'args': {'a': 3, 'b': 10}, 'id': 'call_19zlosKiUbhVbJ0iyC8X30Of', 'type': 'tool_call'}], usage_metadata={'input_tok

In [23]:
llm_with_tools.invoke(message).content

'The product of 3 multiplied by 10 is 30.'

In [24]:
multiple.invoke({'name': 'multiple',
 'args': {'a': 10, 'b': 3},
 'id': 'call_YcO3H3bTmJqJqIJau0GXO5E2',
 'type': 'tool_call'})

ToolMessage(content='30', name='multiple', tool_call_id='call_YcO3H3bTmJqJqIJau0GXO5E2')

# Currency Converter Tool In Langchain

In [69]:
# tool Create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/df7745014d40b1725e358ee7/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()

#  Annotated[float, InjectedToolArg] (Iska Simple Mean h If value provide by 
# the User then accept otherwise no need to predicted by LLM )

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculate the target currency value from a given base currency value.
    """

    return base_currency_value * conversion_rate

In [70]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1757030401,
 'time_last_update_utc': 'Fri, 05 Sep 2025 00:00:01 +0000',
 'time_next_update_unix': 1757116801,
 'time_next_update_utc': 'Sat, 06 Sep 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 88.1925}

In [71]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})

851.5999999999999

In [72]:
# Tool Binding 

llm = ChatOpenAI()

In [73]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [74]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [75]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [76]:
ai_message = llm_with_tools.invoke(messages)

In [82]:
messages.append(ai_message)

In [84]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_qaIOMgF5PQU4l319KytO51IK',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': None},
  'id': 'call_qeNb0px71taPIpVTdsrvTxBH',
  'type': 'tool_call'}]

In [86]:
import json

for tool_call in ai_message.tool_calls:
    # Execute the 1st tool and get the value of conversion rate
    # Execute the 2nd Tool using the conversion rate from tool 1

    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print(tool_message1)

        # Fetch this conversion rate
        # print(tool_message1)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']

        messages.append(tool_message1)
        # Append this tool message to message List
    if tool_call['name'] == 'convert':
        # Eftch the Current Arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1757030401, "time_last_update_utc": "Fri, 05 Sep 2025 00:00:01 +0000", "time_next_update_unix": 1757116801, "time_next_update_utc": "Sat, 06 Sep 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 88.1925}' name='get_conversion_factor' tool_call_id='call_qaIOMgF5PQU4l319KytO51IK'


In [87]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1757030401, "time_last_update_utc": "Fri, 05 Sep 2025 00:00:01 +0000", "time_next_update_unix": 1757116801, "time_next_update_utc": "Sat, 06 Sep 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 88.1925}', name='get_conversion_factor', tool_call_id='call_qaIOMgF5PQU4l319KytO51IK'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_qaIOMgF5PQU4l319KytO51IK', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_qeNb0px71taPIpVTdsrvTxBH', 'function': {'arguments': '{"base_currency_value": 10}', '

In [90]:
llm_with_tools.invoke(messages).content

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[1].role', 'code': None}}